<a href="https://colab.research.google.com/github/Lavesh789/Fine-Tune-an-LLM-for-Sanskrit-English/blob/main/Sanskrit_FineTune_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Model: Tesla T4


Install Required Libraries

In [15]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "triton>=3.0.0" bitsandbytes
!pip install datasets trl huggingface_hub accelerate peft sentencepiece

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-w0an1wu0/unsloth_930f172f7e904889973273f44a97a2fd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-w0an1wu0/unsloth_930f172f7e904889973273f44a97a2fd
  Resolved https://github.com/unslothai/unsloth.git to commit 8ee07d6ae61e92544f63a733daf86f9b6a6b0ef2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Load Base Model with QLoRA Adapters

In [16]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

#Load Base Quantized Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

#Explicitly attach LoRA adapters (Required for training quantized models)
model = FastLanguageModel.get_peft_model(
    model,
    #LoRA Rank
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("SUCCESS: Base model loaded and PEFT adapters attached!")

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


SUCCESS: Base model loaded and PEFT adapters attached!


Prepare the Sanskrit-English Dataset

In [17]:
from datasets import Dataset

# Sample Sanskrit-English Dataset
dataset_data = [
    {
        "instruction": "Translate the following Sanskrit text to English.",
        "input": "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन।",
        "output": "You have a right to perform your prescribed duties, but you are never entitled to the fruits of your actions."
    },
    {
        "instruction": "Explain the underlying philosophy of this Sanskrit verse.",
        "input": "विद्या ददाति विनयं विनयाद्याति पात्रताम्।",
        "output": "Knowledge yields humility; from humility comes worthiness. It highlights that true education develops discipline rather than ego."
    },
    {
        "instruction": "Translate the following Sanskrit text to English.",
        "input": "सत्यमेव जयते नानृतम्।",
        "output": "Truth alone triumphs, not falsehood."
    }
]

raw_dataset = Dataset.from_list(dataset_data)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(inst, inp, out) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

formatted_dataset = raw_dataset.map(format_prompts, batched = True)
print("Dataset formatting complete!")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset formatting complete!


Train the Model (Fine-Tuning)

In [18]:
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# 1. Load Base 4-bit Quantized Model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# 2. Prepare Quantized Model for k-bit Training
model = prepare_model_for_kbit_training(model)

# 3. Define Explicit LoRA Config & Attach Adapter
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Attach trainable LoRA adapter to the quantized model
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()  # Verifies trainable adapters are active

# 4. Define Dataset
dataset_data = [
    {
        "instruction": "Translate the following Sanskrit text to English.",
        "input": "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन।",
        "output": "You have a right to perform your prescribed duties, but you are never entitled to the fruits of your actions.",
    },
    {
        "instruction": "Explain the underlying philosophy of this Sanskrit verse.",
        "input": "विद्या ददाति विनयं विनयाद्याति पात्रताम्।",
        "output": "Knowledge yields humility; from humility comes worthiness. It highlights that true education develops discipline rather than ego.",
    },
    {
        "instruction": "Translate the following Sanskrit text to English.",
        "input": "सत्यमेव जयते नानृतम्।",
        "output": "Truth alone triumphs, not falsehood.",
    },
]

raw_dataset = Dataset.from_list(dataset_data)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


def format_prompts(examples):
    texts = []
    for inst, inp, out in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = (
            alpaca_prompt.format(inst, inp, out)
            + getattr(tokenizer, "eos_token", "<|end_of_text|>")
        )
        texts.append(text)
    return {"text": texts}


formatted_dataset = raw_dataset.map(format_prompts, batched=True)

# 5. Fine-Tune with SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

# Execute training
trainer.train()

# Save LoRA adapters
model.save_pretrained("sanskrit_lora_model")
tokenizer.save_pretrained("sanskrit_lora_model")
print("Training finished successfully!")

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,2.353647
20,0.287215
30,0.084248
40,0.025963
50,0.025047
60,0.018363


Training finished successfully!


Clear GPU Memory (Prevents Out-Of-Memory Error)

In [19]:
import gc

del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory cleared successfully!")

GPU Memory cleared successfully!


Run Inference on the Fine-Tuned Model

In [23]:
from unsloth import FastLanguageModel

#Reload the fine-tuned model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "sanskrit_lora_model", # Path to saved LoRA adapters
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

#Enable 2x faster inference engine
FastLanguageModel.for_inference(model)

#Define Test Input
prompt = alpaca_prompt.format(
    "Translate the following Sanskrit verse to English.",
    "सत्यमेव जयते नानृतम्।",
    ""
)

#Generate Output
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)

print("\nINFERENCE RESULT")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load sanskrit_lora_model as a legacy tokenizer.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



INFERENCE RESULT
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the following Sanskrit verse to English.

### Input:
सत्यमेव जयते नानृतम्।

### Response:
Truth alone triumphs, not falsehood.


Generate Deliverable .py Files for Assignment Submission

In [22]:
# Write train.py automatically
with open("train.py", "w") as f:
    f.write('''import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

def main():
    max_seq_length = 2048
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Llama-3.2-1B-Instruct",
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    dataset_data = [
        {"instruction": "Translate the following Sanskrit text to English.", "input": "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन।", "output": "You have a right to perform your prescribed duties, but you are never entitled to the fruits of your actions."}
    ]
    raw_dataset = Dataset.from_list(dataset_data)

    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

    def format_prompts(examples):
        texts = []
        for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
            texts.append(alpaca_prompt.format(inst, inp, out) + tokenizer.eos_token)
        return { "text" : texts }

    formatted_dataset = raw_dataset.map(format_prompts, batched = True)

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = formatted_dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            max_steps = 60,
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs",
        ),
    )
    trainer.train()
    model.save_pretrained("sanskrit_lora_model")
    tokenizer.save_pretrained("sanskrit_lora_model")

if __name__ == "__main__":
    main()
''')

# Write inference.py automatically
with open("inference.py", "w") as f:
    f.write('''import torch
from unsloth import FastLanguageModel

def run_inference():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "sanskrit_lora_model",
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

    prompt = alpaca_prompt.format(
        "Translate the following Sanskrit verse to English.",
        "सत्यमेव जयते नानृतम्।",
        ""
    )

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
    print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

if __name__ == "__main__":
    run_inference()
''')

print("Created 'train.py' and 'inference.py' in your Colab files!")

Created 'train.py' and 'inference.py' in your Colab files!
